📊 SCREENSENSE: KIDS' SCREENTIME VISUALIZATION

🔹ScreenSense is a data analytics project that studies screen time behavior among Indian children.

🔹It analyzes how long kids use screens, which devices they prefer, and related health impacts.

🔹The project uses data cleaning, feature engineering, visualization, and dashboarding techniques.


✅ WEEK 1 INSIGHTS (Data Understanding & Cleaning)

In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [24]:
df = pd.read_csv(r"C:\Users\kanna\OneDrive\Desktop\Kids ScreenTime Visualization\Indian_Kids_Screen_Time.csv")

print("Dataset Loaded Successfully ✅")

Dataset Loaded Successfully ✅


In [25]:
df.head()

,Age,Gender,Avg_Daily_Screen_Time_hr,Primary_Device,Exceeded_Recommended_Limit,Educational_to_Recreational_Ratio,Health_Impacts,Urban_or_Rural
0,14,Male,3.99,Smartphone,True,0.42,"Poor Sleep, Eye Strain",Urban
1,11,Female,4.61,Laptop,True,0.30,Poor Sleep,Urban
2,18,Female,3.73,TV,True,0.32,Poor Sleep,Urban
3,15,Female,1.21,Laptop,False,0.39,NaN,Urban
4,12,Female,5.89,Smartphone,True,0.49,"Poor Sleep, Anxiety",Urban


In [26]:
df.shape

(9712, 8)

In [27]:
df.columns

Index(['Age', 'Gender', 'Avg_Daily_Screen_Time_hr', 'Primary_Device',
       'Exceeded_Recommended_Limit', 'Educational_to_Recreational_Ratio',
       'Health_Impacts', 'Urban_or_Rural'],
      dtype='object')

In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9712 entries, 0 to 9711
Data columns (total 8 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Age                                9712 non-null   int64  
 1   Gender                             9712 non-null   object 
 2   Avg_Daily_Screen_Time_hr           9712 non-null   float64
 3   Primary_Device                     9712 non-null   object 
 4   Exceeded_Recommended_Limit         9712 non-null   bool   
 5   Educational_to_Recreational_Ratio  9712 non-null   float64
 6   Health_Impacts                     6494 non-null   object 
 7   Urban_or_Rural                     9712 non-null   object 
dtypes: bool(1), float64(2), int64(1), object(4)
memory usage: 540.7+ KB


In [29]:
df.duplicated().sum()

44

In [30]:
df.describe()

,Age,Avg_Daily_Screen_Time_hr,Educational_to_Recreational_Ratio
count,9712.000000,9712.000000,9712.000000
mean,12.979201,4.352837,0.427226
std,3.162437,1.718232,0.073221
min,8.000000,0.000000,0.300000
25%,10.000000,3.410000,0.370000
50%,13.000000,4.440000,0.430000
75%,16.000000,5.380000,0.480000
max,18.000000,13.890000,0.600000


1️⃣ Remove Duplicates

In [31]:
df = df.drop_duplicates()
print("Duplicates removed ✅")

Duplicates removed ✅


2️⃣ Handle Missing Values

In [32]:
# Fill missing age with median
if 'age' in df.columns:
    df['age'].fillna(df['age'].median(), inplace=True)

# Drop rows if screentime is missing
if 'daily_hours' in df.columns:
    df = df.dropna(subset=['daily_hours'])

print("Missing values handled ✅")

Missing values handled ✅


3️⃣ Standardize Categorical Columns

In [33]:
# Convert categorical columns to lowercase and strip spaces
categorical_cols = ['gender', 'location_type', 'device_type', 'activity_category']

for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].str.lower().str.strip()

print("Categories standardized ✅")

Categories standardized ✅


4️⃣ Fix Date Column

In [34]:
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    print("Date column formatted ✅")

In [35]:
df.columns = df.columns.str.strip().str.lower()

In [36]:
print(df.columns)


Index(['age', 'gender', 'avg_daily_screen_time_hr', 'primary_device',
       'exceeded_recommended_limit', 'educational_to_recreational_ratio',
       'health_impacts', 'urban_or_rural'],
      dtype='object')


⚙️ ✅ WEEK 2 INSIGHTS (Feature Engineering / Derived Columns)

1️⃣ Age Band

In [37]:
df['age_band'] = pd.cut(
    df['age'],
    bins=[0,5,10,13,18],
    labels=[
        '0-5 (Early Childhood)',
        '6-10 (Primary)',
        '11-13 (Middle)',
        '14-18 (Teen)'
    ]
)

2️⃣ Usage Level

In [38]:
df['usage_level'] = pd.cut(
    df['avg_daily_screen_time_hr'],
    bins=[0,2,5,24],
    labels=['Low Usage','Moderate Usage','High Usage']
)


3️⃣ Risk Category

In [39]:
df['risk_category'] = pd.cut(
    df['avg_daily_screen_time_hr'],
    bins=[0,3,6,24],
    labels=['Safe','Watch','Risk']
)

In [40]:
df['is_heavy_user'] = df['avg_daily_screen_time_hr'] > 5

4️⃣ Dependency Index

In [41]:
max_hours = df['avg_daily_screen_time_hr'].max()

df['dependency_index'] = df['avg_daily_screen_time_hr'] / max_hours

In [42]:
df['device_popularity'] = df.groupby('primary_device')['primary_device'].transform('count')

5️⃣ Educational Dominance

In [43]:
df['learning_type'] = df['educational_to_recreational_ratio'].apply(
    lambda x: 'More Educational' if x >= 0.5 else 'More Recreational'
)

6️⃣ Urban Usage Intensity

In [44]:
df['urban_usage_flag'] = (
    (df['urban_or_rural'] == 'Urban') & 
    (df['avg_daily_screen_time_hr'] > 5)
)